# Vaccine Hesitancy Prediction — Logistic Regression

> **Research:** Uncovering determinants of vaccine hesitancy in India: A comparative machine learning framework for data-driven insights  
> **Conference:** AAAI Student Chapter — Young Researchers' Conference 2025, PCCOE

This notebook implements **Logistic Regression** with GridSearchCV hyperparameter tuning and SMOTE class balancing to predict vaccine hesitancy from structured survey data.

---

**Pipeline:**
1. Load & clean survey data
2. Feature encoding (Ordinal + One-Hot)
3. Feature scaling (StandardScaler)
4. Class balancing (SMOTE)
5. Train/test split
6. Logistic Regression + GridSearchCV
7. Evaluation (Accuracy, Classification Report, Confusion Matrix)

## 1. Imports

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

print("All libraries loaded successfully.")

## 2. Load Dataset

The dataset is a cleaned version of the primary survey responses collected from Indian respondents.  
Update the path below to match your local file location.

In [ ]:
# Update this path to your dataset location
df = pd.read_csv("../data/Vaccine_Hesitancy_Research_Clean.csv")

# Strip whitespace from column names
df.columns = df.columns.str.strip()

print(f"Dataset shape: {df.shape}")
df.head()

## 3. Feature Selection

18 features were selected across four categories:
- **Socio-demographic**: Age, Gender, Area, Education, Employment, Income
- **Health background**: Chronic conditions, COVID history, Healthcare proximity, Prior vaccinations
- **Trust factors**: Trust in doctors, government, friends/family
- **Attitudes**: Safety beliefs, effectiveness beliefs, side-effect worry, wait-and-see tendency
- **Motivation**: What would encourage vaccination

In [ ]:
selected_features = [
    "Age",
    "Gender",
    "Which type of area do you live in?",
    "Education",
    "Employment status",
    "Monthly household income",
    "Do you or your family members have any chronic health conditions?",
    "Have you or a close family member been infected with COVID-19 in the past?",
    "Do you or someone close to you work in healthcare?",
    "Have you taken all the previous vaccines (polio, BCG, MMR,  Hepatitis B vaccine, DTP, RVV)  offered by the Indian Govt. ?",
    "How much do you trust the following sources for information about vaccines? [Doctors/Healthcare workers]",
    "How much do you trust the following sources for information about vaccines? [Government (Health Ministry / Public Health)]",
    "How much do you trust the following sources for information about vaccines? [Friends & family]",
    "Please indicate your level of agreement with the following statements: [Vaccines are safe.]",
    "Please indicate your level of agreement with the following statements: [Vaccines are effective in preventing serious illness.]",
    "Please indicate your level of agreement with the following statements: [I am worried about potential side effects from the vaccine.]",
    "Please indicate your level of agreement with the following statements: [I prefer to wait and see how the vaccine affects others before getting it.]",
    "What would make you more likely to get vaccinated? (Select all that apply)"
]

target_col = "Did you ever feel hesitant before taking the vaccine?"

df = df[selected_features + [target_col]]

print(f"Target distribution:\n{df[target_col].value_counts()}")

## 4. Preprocessing

- **OrdinalEncoder** for Education and Income (these have a natural order)
- **One-Hot Encoding** (`pd.get_dummies`) for remaining categorical features
- **StandardScaler** to normalize all features (important for Logistic Regression)

In [ ]:
X = df[selected_features].copy()
y = df[target_col]

# Ordinal encoding for features with natural ordering
ordinal_features = ["Education", "Monthly household income"]
encoder = OrdinalEncoder()
X[ordinal_features] = encoder.fit_transform(X[ordinal_features])

# One-hot encoding for remaining categorical features
X = pd.get_dummies(X, drop_first=True)

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features after encoding: {X.shape[1]}")

## 5. Handle Class Imbalance with SMOTE

**SMOTE (Synthetic Minority Over-sampling Technique)** generates synthetic samples for the minority class (Hesitant) to balance the dataset. This prevents the model from being biased toward predicting the majority class.

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)

print(f"Before SMOTE: {dict(y.value_counts())}")
print(f"After SMOTE:  {dict(pd.Series(y_resampled).value_counts())}")

## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.2,
    random_state=42,
    stratify=y_resampled
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

## 7. Logistic Regression + GridSearchCV

We tune three hyperparameters:
- `C`: Regularization strength (lower = stronger regularization)
- `penalty`: L1 (Lasso) or L2 (Ridge) regularization
- `solver`: Optimization algorithm

In [ ]:
params = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga"]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight="balanced"),
    param_grid=params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Hyperparameters:", grid.best_params_)
print(f"Best CV Accuracy: {grid.best_score_:.4f}")

## 8. Evaluation

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 9. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test,
    display_labels=best_model.classes_,
    cmap="Blues",
    ax=ax
)
plt.title("Confusion Matrix — Logistic Regression", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("../results/confusion_matrix_lr.png", dpi=150)
plt.show()

---

## Summary

| Metric | Value |
|--------|-------|
| Accuracy | ~75.5% |
| Precision | 0.79 |
| Recall | 0.75 |
| F1-Score | 0.72 |

**Observation:** Logistic Regression provides interpretable coefficients useful for policy insights, but its assumption of linear relationships limits performance compared to ensemble methods. See `random_forest_xgboost.ipynb` for higher accuracy results.